# 04: Conversation memory and delayed recall

**Question:** Does the answer to a final request use requirements retained from earlier turns?

**Inputs:** Seeded conversations containing constraints, distractors and tool requests. The default scripted backend receives only the current message and records retrieved from Reckon.

**Outputs:** Final-answer constraint recall, tool-name coverage, transcripts with retrieved IDs and prompt context, operation events, policy comparisons and a paired context-withholding control.

The scripted fixture recognizes this environment's templates. It quotes remembered requirements; it does not demonstrate general language understanding or compliance with every requested prose format. Optional Ollama calls use the same memory prompt.


## 1. Setup


In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]
BACKEND = "sim"  # @param ["sim", "ollama"]


In [ ]:
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import (
    NotebookConfig,
    print_config,
    print_repro_stamp,
    set_all_seeds,
    RunManager,
    bootstrap_ci,
    paired_permutation_test,
    effect_size,
    setup_plotting,
    add_caption,
    COLORS,
    POLICY_COLORS,
)
from _utils.adapters import (
    ToolEnv,
    ToolEnvConfig,
    Episode,
    EpisodeMetrics,
    ollama_healthcheck,
    ollama_chat,
    ollama_list_models,
    parse_agent_response,
    OLLAMA_AVAILABLE,
    SimulatedOllamaBackend,
    AGENT_SYSTEM_PROMPT,
)

from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

import matplotlib.pyplot as plt
import numpy as np
import random
from dataclasses import dataclass, asdict
from typing import Optional, Any

setup_plotting()
print("Imports OK!")
from reckon.metrics.tests.paired import paired_inference


In [ ]:
# Configuration
config = NotebookConfig(
    notebook_name="04_conversation_simulator",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=20 if RUN_MODE == "paper" else 5,
    budget_slots=20,
    task_name="conversation",
    num_episodes=20 if RUN_MODE == "paper" else 5,
    evidence_weight=0.5,
    custom={
        "backend": BACKEND,
        "fixture_version": 3,
        "n_steps_per_episode": 8,
    },
)
print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)


In [ ]:
# Preserve the requested backend; each runner counts actual fallback calls.
print(f"Requested backend: {BACKEND}; Ollama available: {OLLAMA_AVAILABLE}")
if BACKEND == "ollama" and not OLLAMA_AVAILABLE:
    print("Model calls will use the scripted fallback and increment fallback_count.")


## 2. Problem and hypotheses

The current message is insufficient for a delayed request such as “follow all my requirements.” The agent must keep the earlier requirements, retrieve them and use them in its answer. Distractors and tool results compete for memory.

The policy comparison asks whether final-answer recall changes under that pressure. Each run starts with fresh memory. Initial utilities express the caller's priorities; `observe` then shares the turn's outcome credit among retrieved records. EvidencePerBytePolicy has no measured effects here and uses the same utility branch as UtilityPolicy. A policy name does not establish an ablation effect.

A paired control uses the same seed, budget and scripted backend with retrieved context withheld. It tests the whole memory-input path; it does not identify the effect of any one record. Notebook 03 demonstrates the native single-record evaluator separately.


## 3. Minimal Theory

### Agent Protocol

Each agent response must be strict JSON:

```json
{
  "tool": null,
  "final": "Response to user",
  "memory_write": [{"tag": "constraint", "content": "use JSON format"}]
}
```

### Memory-Augmented Conversation Loop

At each step:

1. Write the current user observation.
2. Retrieve constraint and tool-result records into the turn's credit context.
3. Compose the prompt from those retrieved records and the current message.
4. Call the backend and parse its response.
5. Write any memories proposed by the response; these writes can prune records.
6. Step the environment, which executes any requested tool and returns the outcome.
7. Pass that outcome to `observe` for the retrieved context; records already pruned cannot receive a stored utility update.
8. Store the returned tool result, when present.
9. Record the retrieved IDs, prompt context, response, tool result and reward in the transcript.

### What the scores mean

- `constraint_retention` is literal recall of required values in the final answer. Earlier acknowledgements and emitted memory writes do not count. This is not a check that the answer obeys each prose constraint.
- `tool_correctness` measures expected/called tool-name overlap, not the correctness of arguments or later use of results.
- `contradiction_rate` remains an unimplemented diagnostic: the environment does not populate its violation flags.
- `final_success` is the environment's composite threshold, additionally requiring no notebook parse failures. It is not an independent user outcome.

Every transcript turn includes the actual retrieved IDs and memory prompt. `use_memory=False` withholds that prompt while retaining the same write path. The control is not a zero-storage baseline.


## 4. System Mapping

```mermaid
flowchart TB
    subgraph ENV["Tool Environment"]
        direction LR
        E1["User: States constraint"]
        E2["User: Red herring"]
        E3["User: Tool request"]
        E4["User: Reference earlier"]
        E5["User: Final goal"]
        E1 --> E2 --> E3 --> E4 --> E5
    end

    subgraph AGENT["Agent Loop"]
        A1["Receive observation"]
        A2["Query CMM for context"]
        A3["Compose prompt"]
        A4["Call backend (sim/ollama)"]
        A5["Parse JSON response"]
        A6["Write proposed memories"]
        A7["Step environment and execute tool"]
        A8["Observe outcome for retrieved context"]
        A9["Store tool result"]
        A1 --> A2 --> A3 --> A4 --> A5 --> A6 --> A7 --> A8 --> A9
    end

    subgraph CMM["CausalMemoryManager"]
        M1["retrieve(query, limit=k)"]
        M2["write(payload, tags)"]
        M3["prune() under budget"]
    end

    ENV --> AGENT
    AGENT --> CMM
    CMM --> AGENT
```


## 5. Experiment Design

**Unit of analysis:** One conversation episode

**Episode structure:**

- 8 user messages per episode
- Two or three generated constraints
- Distractors
- Tool requests
- References when the configured message count leaves room
- Final goal requiring all constraints

**Comparison:**

- Policies: FIFO, LRU, Utility, Evidence per byte
- Budgets: 5, 10, 15, 20 slots
- Seed labels: 5 (demo) / 20 (paper), one per episode in each policy/budget condition

**Metrics:**

- Final-answer literal constraint recall (primary)
- tool_use_correctness
- contradiction_rate
- final_goal_success

**Statistics (paper mode):**

- Paired permutation test across seeds
- Normal-approximation intervals for paired mean differences

Seeds are joined explicitly before each paired comparison. These tests are unadjusted across budgets. Mixed Ollama/fallback runs remain labelled in the table; the scripted control does not validate a live model.


## 6. Demo Run


In [ ]:
# Create environment
tool_env = ToolEnv(
    ToolEnvConfig(
        n_steps=config.custom["n_steps_per_episode"],
        require_strict_json=True,
    )
)

# Preview an episode
obs, info = tool_env.reset(seed=config.seed)
episode = tool_env.get_episode()

print(f"Episode {episode.id}")
print(f"Constraints: {episode.required_constraints}")
print(f"Expected tools: {episode.expected_tool_calls}")
print(f"\nMessages:")
for i, msg in enumerate(episode.messages):
    marker = ""
    if msg.is_constraint:
        marker = " [CONSTRAINT]"
    elif msg.is_red_herring:
        marker = " [RED HERRING]"
    elif msg.requires_tool:
        marker = f" [TOOL: {msg.requires_tool}]"
    elif msg.is_reference:
        marker = " [REFERENCE]"
    print(
        f"  {i}. {msg.content[:60]}...{marker}"
        if len(msg.content) > 60
        else f"  {i}. {msg.content}{marker}"
    )


In [ ]:
from _utils.adapters.conversation import run_conversation_episode


In [ ]:
# Demo run
demo_result = run_conversation_episode(
    policy=EvidencePerBytePolicy(config.evidence_weight),
    budget_slots=15,
    seed=config.seed,
    backend=BACKEND,
    n_steps=config.custom["n_steps_per_episode"],
    verbose=True,
)

print(f"\nDemo Episode Results:")
print(f"  Final-answer constraint recall: {demo_result['constraint_retention']:.2%}")
print(f"  Tool-name coverage: {demo_result['tool_correctness']:.2%}")
print(f"  Contradiction rate: {demo_result['contradiction_rate']:.2%}")
print(f"  Final success: {demo_result['final_success']}")
print(f"  Parse failures: {demo_result['parse_failures']}")
print(f"  Final memory slots: {demo_result['final_slots']}")


## 7. Full Run


In [ ]:
# Policy and budget configurations
POLICIES = {
    "FIFO": FIFOPolicy(),
    "LRU": LRUPolicy(),
    "Utility": UtilityPolicy(),
    "Evidence per byte": EvidencePerBytePolicy(config.evidence_weight),
}

BUDGETS = [5, 10, 15, 20]

print(f"Running {config.num_episodes} episodes per configuration")
print(f"Policies: {list(POLICIES.keys())}")
print(f"Budgets: {BUDGETS}")
print(f"Backend: {BACKEND}")


In [ ]:
# Run experiments
all_results = []
all_transcripts = []

total_runs = len(POLICIES) * len(BUDGETS) * config.num_episodes
run_count = 0

for policy_name, policy in POLICIES.items():
    for budget in BUDGETS:
        for ep in range(config.num_episodes):
            seed = config.seed + ep
            result = run_conversation_episode(
                policy=policy,
                budget_slots=budget,
                seed=seed,
                backend=BACKEND,
                n_steps=config.custom["n_steps_per_episode"],
            )

            # Store result (without transcript for aggregation)
            result_summary = {
                k: v for k, v in result.items() if k not in ["transcript", "events"]
            }
            all_results.append(result_summary)

            # Store transcript separately
            all_transcripts.append(
                {
                    "policy": policy_name,
                    "budget": budget,
                    "seed": seed,
                    "transcript": result["transcript"],
                }
            )

            run_count += 1
            if run_count % 20 == 0:
                print(f"Progress: {run_count}/{total_runs}")

print(f"\nCompleted {len(all_results)} runs")


In [ ]:
from statistics import mean, stdev


def scores_for(policy_name, budget, metric):
    return {
        r["seed"]: r[metric]
        for r in all_results
        if r["policy"] == policy_name and r["budget"] == budget
    }


summary = []
for policy in POLICIES.values():
    for budget in BUDGETS:
        scores = list(
            scores_for(type(policy).__name__, budget, "constraint_retention").values()
        )
        row = {
            "policy": type(policy).__name__,
            "budget": budget,
            "recall_mean": mean(scores),
            "recall_std": stdev(scores),
            "episodes": len(scores),
        }
        summary.append(row)
        print(row)


## 8. Results


In [ ]:
# Plot: Constraint retention by policy and budget
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Constraint retention
ax = axes[0]
for i, policy_name in enumerate(POLICIES.keys()):
    values = [
        list(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "constraint_retention"
            ).values()
        )
        for budget in BUDGETS
    ]
    means = [mean(v) for v in values]
    stds = [stdev(v) for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Final-answer Constraint Recall")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Final-answer Constraint Recall vs Budget")

# Right: Tool correctness
ax = axes[1]
for i, policy_name in enumerate(POLICIES.keys()):
    values = [
        list(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "tool_correctness"
            ).values()
        )
        for budget in BUDGETS
    ]
    means = [mean(v) for v in values]
    stds = [stdev(v) for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Expected Tool-Name Coverage")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Expected Tool-Name Coverage vs Budget")

add_caption(
    axes[0],
    "Environment scores by configured policy and budget; no ablation claim",
    n=config.num_episodes * len(BUDGETS),
    mode=RUN_MODE,
)

plt.tight_layout()
plt.show()


In [ ]:
paired_comparisons = []
if RUN_MODE == "paper":
    for budget in BUDGETS:
        baseline = scores_for("FIFOPolicy", budget, "constraint_retention")
        treatment = scores_for("EvidencePerBytePolicy", budget, "constraint_retention")
        assert baseline.keys() == treatment.keys()
        differences = [treatment[seed] - baseline[seed] for seed in sorted(baseline)]
        p_value, interval = paired_inference(
            differences, "permutation", 10000, 0.05, config.seed
        )
        paired_comparisons.append(
            {
                "budget": budget,
                "pairs": len(differences),
                "mean_delta": mean(differences),
                "p_value": p_value,
                "normal_interval": list(interval),
                "adjustment": "none",
            }
        )
        print(
            f"Budget {budget}: {len(differences)} pairs, mean delta {mean(differences):.3f}, p={p_value:.4g}, normal-approximation interval={interval}"
        )
    print(
        "Unadjusted tests across budgets; both policies use utility, not measured retention effects."
    )
else:
    print("Paper mode adds paired inference across the configured seeds.")


In [ ]:
# Heatmap: Success rate by policy and budget
policy_labels = [type(policy).__name__ for policy in POLICIES.values()]
success_grid = np.array(
    [
        [
            mean(scores_for(label, budget, "final_success").values())
            for budget in BUDGETS
        ]
        for label in policy_labels
    ]
)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(success_grid, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(BUDGETS)))
ax.set_xticklabels(BUDGETS)
ax.set_yticks(range(len(policy_labels)))
ax.set_yticklabels(policy_labels)
ax.set_xlabel("Budget (slots)")
ax.set_ylabel("Policy")
ax.set_title("Final Goal Success Rate")

# Add text annotations
for i in range(len(policy_labels)):
    for j in range(len(BUDGETS)):
        val = success_grid[i, j]
        color = "white" if val < 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=12)

plt.colorbar(im, label="Success Rate")
plt.tight_layout()
plt.show()


### Inspect a tight-budget conversation


In [ ]:
failure_result = run_conversation_episode(
    FIFOPolicy(),
    5,
    config.seed,
    backend=BACKEND,
    n_steps=config.custom["n_steps_per_episode"],
)
last_turn = failure_result["transcript"][-1]
print("FIFO, 5 slots: final-answer recall", failure_result["constraint_retention"])
print("Retrieved IDs:", last_turn["retrieved_ids"])
print("Memory prompt:", last_turn["memory_context"])
print("Final answer:", last_turn["agent_response"]["final"])


In [ ]:
# Compare: Same conversation with EvidencePerBytePolicy
success_result = run_conversation_episode(
    policy=EvidencePerBytePolicy(config.evidence_weight),
    budget_slots=5,  # Same tiny budget
    seed=config.seed,  # Same seed
    backend=BACKEND,
    n_steps=config.custom["n_steps_per_episode"],
    verbose=False,
)

print("\nComparison: EvidencePerBytePolicy with same budget")
print("=" * 60)
print(f"Policy: Evidence per byte | Budget: 5 slots")
print(f"Final-answer constraint recall: {success_result['constraint_retention']:.0%}")
print(f"Contradiction rate: {success_result['contradiction_rate']:.0%}")
print(f"Final success: {success_result['final_success']}")
print()

# The difference
fifo_retention = failure_result["constraint_retention"]
evidence_retention = success_result["constraint_retention"]
if evidence_retention > fifo_retention:
    print(
        f"Improvement: +{(evidence_retention - fifo_retention):.0%} constraint retention"
    )
    print("Observed score difference; this comparison does not isolate its cause.")


The transcript connects each answer to the actual records supplied at that turn. The tight-budget comparison can expose a missed requirement; the controlled experiment below isolates whether supplying retained context changes the scripted answer. It does not isolate an individual record's effect or create a debugger graph edge.


### Withhold retrieved context at the decision

Both arms use fresh memory and the same generated conversation. The scripted backend has no hidden conversation cache. The 100-slot allowance fits these eight-turn fixtures; the check verifies there was no pruning. Turning memory input off leaves writes enabled but supplies no context to the response function.


In [ ]:
context_controls = []
for seed in range(config.seed, config.seed + config.seed_count):
    for use_memory in [True, False]:
        result = run_conversation_episode(
            FIFOPolicy(),
            100,
            seed,
            backend="sim",
            n_steps=config.custom["n_steps_per_episode"],
            use_memory=use_memory,
        )
        assert not any(event.get("op_type") == "prune" for event in result["events"])
        assert result["constraint_retention"] == float(use_memory)
        context_controls.append(result)
print(
    f"{len(context_controls)} executed control episodes: retained context recalls all constraints; withheld context recalls none."
)
print("This whole-context control does not estimate individual record effects.")


## 9. Debugger Walkthrough

After exporting the demo's `logs/events.jsonl`, follow the [native debugger quick start](../src/reckon/viz/README.md#quick-start). Inspect operation types, record IDs, read-hit IDs and prune reasons, alongside the separately saved transcript. The minimal CMM operation log does not preserve the full prompts, queries, tags or payloads and does not by itself produce a MEASURED graph edge.

Only the demo event buffer is exported here. Comparison metrics aggregate other runs; they must not be presented as if the imported demo log contained all of them.


## 10. Limits and next steps

The scripted backend recognizes generated templates and quotes requirements found in its memory prompt. Final-answer literal recall is useful for checking the read path; it is not a language-model quality benchmark or proof of prose compliance. Tool-name coverage and the unpopulated contradiction diagnostic retain the limits described above.

Ollama receives current observations plus retrieved context and may fall back to the same scripted backend. Each run reports `fallback_count`; a mixed run must not be presented as pure model inference. The local verification exercises the scripted path and a stubbed model path, not a live Ollama model.

Each episode reports elapsed execution time. The next section runs a separate single-record ablation and charges its full cost. Applying measured effects to the main comparison’s live retention remains unfinished; its policy scores are supplied priorities and learned utility.


## Remove one record at its first read

This separate experiment runs fresh seed-matched episodes under FIFO with enough storage to reach the selected read. It removes one logical record immediately before its first eligible retrieval, verifies matching record state in both arms, and charges both episodes and the full evaluation time. The target is the first constraint record returned by a read in this simulated conversation. Later replacement records remain allowed. These measurements do not score the main comparison's live store or implement measured retention.

The interval is a normal approximation across paired episode differences; the sign-permutation test is unadjusted across record labels. Four demo pairs and twenty paper-mode pairs demonstrate execution; neither count was selected for statistical power. The probe uses the simulated backend; model-backed runs do not supply this paired evidence. An absent or ambiguous target raises an error rather than becoming a zero effect.

In [ ]:
from _utils.adapters.record_probe import AdapterProbeConfig, evaluate_adapter_record

record_probe = None
if BACKEND == "sim":
    record_probe = evaluate_adapter_record(
        AdapterProbeConfig("conversation", budget_slots=100, max_steps=8),
        "first_constraint",
        lambda record: "constraint" in record.tags,
        n_trials=20 if RUN_MODE == "paper" else 4,
        seed_base=1000,
    )
    print({key: record_probe[key] for key in ["protocol", "probe_episodes", "probe_elapsed_s", "confidence_interval"]})
    print({key: value for key, value in record_probe["evaluation"].items() if key != "trials"})
else:
    print("Record probe unavailable: paired evidence requires the declared simulated or live scripted controller.")

## Export Artifacts


In [ ]:
run = RunManager(config)
run.save_config()
if record_probe is not None:
    (run.run_dir / "record_probe.json").write_text(json.dumps(record_probe, indent=2) + "\n")

# Save metrics
run.save_metrics(all_results, filename="metrics.csv")

# Save transcripts as JSONL
run.save_jsonl(all_transcripts, "transcripts.jsonl")

run.save_jsonl(context_controls, "context_controls.jsonl")
run.save_jsonl(paired_comparisons, "paired_comparisons.jsonl")

# Save event logs from demo run
if demo_result.get("events"):
    run.save_logs(
        demo_result["events"]
    )  # Native directory importer looks for events.jsonl.

# Save figures
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Constraint retention
ax = axes[0]
for policy_name in POLICIES.keys():
    means = [
        mean(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "constraint_retention"
            ).values()
        )
        for budget in BUDGETS
    ]
    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.plot(BUDGETS, means, "o-", label=policy_name, color=color)
ax.set_xlabel("Budget")
ax.set_ylabel("Final-answer Constraint Recall")
ax.legend()

# Tool correctness
ax = axes[1]
for policy_name in POLICIES.keys():
    means = [
        mean(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "tool_correctness"
            ).values()
        )
        for budget in BUDGETS
    ]
    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.plot(BUDGETS, means, "o-", label=policy_name, color=color)
ax.set_xlabel("Budget")
ax.set_ylabel("Expected Tool-Name Coverage")
ax.legend()

run.save_figure(fig, "conversation_metrics")
plt.close(fig)

# Save environment metadata
env_meta = {
    "backend": BACKEND,
    "ollama_available": OLLAMA_AVAILABLE,
    "n_steps_per_episode": config.custom["n_steps_per_episode"],
    "fixture_version": 3,
    "outcome": "literal constraint recall in final answer",
    "measured_retention_effects": False,
    "policies": list(POLICIES.keys()),
    "budgets": BUDGETS,
}
with open(run.run_dir / "env_meta.json", "w") as f:
    json.dump(env_meta, f, indent=2)

# Summary
evidence_mean = mean(
    r["constraint_retention"] for r in all_results if r["policy"] == "EvidencePerBytePolicy"
)
fifo_mean = mean(
    r["constraint_retention"] for r in all_results if r["policy"] == "FIFOPolicy"
)

import hashlib
import subprocess

source_paths = [
    NOTEBOOK_DIR / "_utils/adapters/record_probe.py",
    NOTEBOOK_DIR / "_utils/adapters/episode_memory.py",
    NOTEBOOK_DIR / "04_agentic_conversation_simulator.ipynb",
    NOTEBOOK_DIR / "_utils/adapters/conversation.py",
    NOTEBOOK_DIR / "_utils/adapters/ollama_client.py",
    NOTEBOOK_DIR / "_utils/adapters/tool_env.py",
    NOTEBOOK_DIR / "_utils/artifacts.py",
]
source = {
    "run_mode": RUN_MODE,
    "uncommitted_source": bool(
        subprocess.run(
            [
                "git",
                "status",
                "--porcelain",
                "--",
                *[str(path) for path in source_paths],
            ],
            capture_output=True,
            text=True,
            check=True,
        ).stdout
    ),
    "sha256": {
        str(path.relative_to(NOTEBOOK_DIR)): hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        for path in source_paths
    },
}
(run.run_dir / "source.json").write_text(json.dumps(source, indent=2) + "\n")
run.write_summary(
    {
        "run_mode": RUN_MODE,
        "backend": BACKEND,
        "comparison_episodes": len(all_results),
        "context_control_episodes": len(context_controls),
        "record_probe_episodes": record_probe["probe_episodes"] if record_probe else 0,
        "record_probe_elapsed_s": record_probe["probe_elapsed_s"] if record_probe else 0,
        "record_probe_delta": record_probe["evaluation"]["delta"] if record_probe else None,
        "record_probe_available": record_probe is not None,
        "inspection_episodes": 3,
        "score_meaning": "literal constraint recall in final answer",
        "evidence_constraint_retention": evidence_mean,
        "fifo_constraint_retention": fifo_mean,
        "improvement": evidence_mean - fifo_mean,
    }
)

run.print_artifacts()
